# TechQA 数据审计与分组划分

复核固定输入、别名和组件划分。此队列是自定义有答案检索探索实验，不能称为官方 TechQA 测试集。

In [1]:
from pathlib import Path
import json, sys, os
import pandas as pd
from IPython.display import display, Image
root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'project/src/ragstudy').is_dir())
for folder in ('.vendor', '.vendor-hf', '.vendor-onnx-compat'):
    path = root / 'project' / folder
    if path.is_dir():
        sys.path.insert(0, str(path))
sys.path.insert(0, str(root / 'project/src'))
artifacts = sorted(path for path in (root / 'project/artifacts').iterdir()
                   if (path / 'results.json').is_file() and (path / 'search_bundle.joblib').is_file())
assert artifacts, 'First run python project/scripts/run_experiment.py --backend lsa'
artifact = Path(os.environ['RAG_ARTIFACT']).resolve() if os.environ.get('RAG_ARTIFACT') else artifacts[-1]
result = json.loads((artifact / 'results.json').read_text(encoding='utf-8'))
print('Actual local run:', artifact.name, '| backend:', result['retrieval_metadata']['semantic']['method'])

Actual local run: frozen_e2324b7a967c5dbe2766ecc3 | backend: frozen_onnx_fp32_cls_cosine


In [2]:
audit = result['data_audit']
display(pd.DataFrame(audit['splits']).T)
display({key: audit[key] for key in ['raw_documents','unique_documents','canonical_answerable_queries','independent_components','head_words','source_hashes']})

,queries,groups
train,410,336
dev,84,72
test,84,72


{'raw_documents': 28481,
 'unique_documents': 28478,
 'canonical_answerable_queries': 578,
 'independent_components': 480,
 'head_words': 256,
 'source_hashes': {'train.json': '69d97231509482ed6bd5ec1c4bc0607acb82a88d11169eb8383592d0ca8b93c7',
  'corpus.zip': 'c06aa287dcc1abf8db6b49b8495df095db73342d729f6451ac330785245d10be'}}

In [3]:
queries = json.loads((artifact / 'split_manifest.json').read_text(encoding='utf-8'))
frame = pd.DataFrame(queries)
assert frame.groupby('group_id')['split'].nunique().max() == 1
print('Group-disjoint:', True)
print('Automated near-duplicate pairs:', len(audit['near_duplicate_pairs']))
print('Human review:', audit['semantic_duplicate_human_review'])

Group-disjoint: True
Automated near-duplicate pairs: 54
Human review: pending; automated grouping is not exhaustive
